# When rows aren't IID: honest validation with `validation_structure`

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Innixma/kdd2026_tutorial_materials/blob/main/notebooks/06_noniid_validation.ipynb)
[![GitHub Repo](https://img.shields.io/badge/GitHub-Repo-181717?logo=github)](https://github.com/Innixma/kdd2026_tutorial_materials)
[![Tutorial Website](https://img.shields.io/badge/Tutorial-Website-0a7aca?logo=googlechrome&logoColor=white)](https://kdd26-automl-hands-on.github.io/)

**Taming Structured Data Foundation Models with AutoML — KDD 2026 hands-on tutorial**

Every validation split so far assumed rows are independent. Much real tabular data is not:
repeated measurements per patient, transactions per customer, readings over time. Split
such data randomly and the "held-out" rows share their entity with training rows — the
model recognizes *the mouse*, not the biology, and the validation score becomes fiction.

The dataset: **mice protein expression** — 72 mice, 15 measurements each (1,080 rows),
77 protein levels, 8 classes. New data in deployment means *new mice*, so honest
evaluation must hold out whole mice. We build one honest, mouse-disjoint test set, then
fit AutoGluon twice on the identical training data:

1. **Naive** — the default random holdout, as if rows were IID.
2. **Grouped** — `validation_structure={"group_on": "mouse"}`, which makes every internal
   split mouse-disjoint.

> **Runtime**: ~2 minutes, CPU only.

## Setup

In [1]:
# Installs everything the notebook needs (fast via uv; a no-op where already present).
import sys
!command -v uv >/dev/null || pip install -q uv
!uv pip install -q --python {sys.executable} autogluon.tabular openml

import numpy as np
import openml
from autogluon.tabular import TabularPredictor

## The data, and an honest test set

OpenML's copy carries the measurement id (`MouseID`, e.g. `309_1` … `309_15`); the part
before the underscore identifies the mouse. We hold out 30% of the *mice* — not 30% of the
rows — as the test set both runs share.

In [2]:
ds = openml.datasets.get_dataset(40966)  # MiceProtein
df, *_ = ds.get_data(include_row_id=True)
df["mouse"] = df["MouseID"].astype(str).str.split("_").str[0]
df = df.drop(columns=["MouseID"])

rng = np.random.default_rng(0)
mice = df["mouse"].unique()
test_mice = set(rng.choice(mice, size=int(0.3 * len(mice)), replace=False))
train = df[~df["mouse"].isin(test_mice)].reset_index(drop=True)
test = df[df["mouse"].isin(test_mice)].reset_index(drop=True)
print(f"{df['mouse'].nunique()} mice, {len(train)} train rows, {len(test)} test rows ({len(test_mice)} held-out mice)")

72 mice, 765 train rows, 315 test rows (21 held-out mice)


## Run 1 — the naive split

We drop the `mouse` column (a naive user wouldn't think of it as a feature or a split key)
and let AutoGluon use its default random holdout.

In [3]:
naive = TabularPredictor(label="class", eval_metric="log_loss", path="noniid_naive", verbosity=0).fit(
    train.drop(columns=["mouse"]),
    hyperparameters={"GBM": {}, "RF": {}, "XGB": {}},
)
naive.leaderboard(test.drop(columns=["mouse"]))

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,RandomForest,-1.127064,-0.302254,log_loss,0.128664,0.065471,0.502577,0.128664,0.065471,0.502577,1,True,2
1,XGBoost,-1.809081,-0.107874,log_loss,0.157179,0.024861,5.868492,0.157179,0.024861,5.868492,1,True,3
2,WeightedEnsemble_L2,-1.893879,-0.070965,log_loss,0.334267,0.093844,12.829453,0.008483,0.000661,0.029798,2,True,4
3,LightGBM,-2.648236,-0.081834,log_loss,0.039941,0.002851,6.428587,0.039941,0.002851,6.428587,1,True,1


Read that leaderboard carefully — it is a double disaster:

- **The scores are fiction.** Validation says log-loss ≈ 0.07; the mouse-disjoint test says
  ≈ 1.9. The estimate is off by more than **25×**, because every "held-out" row had
  siblings from the same mouse in training.
- **Model selection is inverted.** Validation ranks the boosted trees above RandomForest;
  on genuinely new mice, RandomForest is by far the best and the boosted trees are the
  worst. The leaked split doesn't just misestimate — it picks the wrong model.

## Run 2 — declare the structure

One argument fixes both problems: `validation_structure` tells AutoGluon the rows are
grouped by `mouse`, and every internal split — holdout or bagged folds — becomes
group-disjoint. (For temporal data the analogous key is `time_on`; for both at once,
`group_time_on`.)

In [4]:
grouped = TabularPredictor(label="class", eval_metric="log_loss", path="noniid_grouped", verbosity=0).fit(
    train,
    hyperparameters={"GBM": {}, "RF": {}, "XGB": {}},
    validation_structure={"group_on": "mouse"},
)
grouped.leaderboard(test)

,model,score_test,score_val,eval_metric,pred_time_test,pred_time_val,fit_time,pred_time_test_marginal,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,RandomForest,-1.285376,-1.419502,log_loss,0.115767,0.067013,0.442891,0.115767,0.067013,0.442891,1,True,2
1,WeightedEnsemble_L2,-1.522193,-1.308975,log_loss,0.283420,0.076263,39.491888,0.005753,0.000617,0.029500,2,True,4
2,LightGBM,-1.784961,-1.384156,log_loss,0.016890,0.001957,30.829847,0.016890,0.001957,30.829847,1,True,1
3,XGBoost,-2.312233,-1.511217,log_loss,0.145010,0.006676,8.189649,0.145010,0.006676,8.189649,1,True,3


Now validation and test agree: the estimate is honest (within a factor ~1.2 rather than
25×), RandomForest correctly rises in the ranking, and the final selected model scores
substantially better on the honest test set than the naive run's choice — the naive run
didn't just *report* the wrong number, it *shipped* a worse model.

## Takeaways

- If your rows share entities (patients, customers, devices, stores) or arrive over time,
  IID validation silently breaks — and TFMs' in-context memorization makes them at least as
  exposed to this as trees.
- `validation_structure` is declarative: `group_on`, `time_on`, `group_time_on`,
  `stratify_on` — one dict, and holdouts, bagged folds, and ensembling all honor it.
- Benchmarks need this too: [TabArena](https://tabarena.ai)'s BeyondArena tasks carry
  grouped and temporal splits as first-class citizens, and leaderboards there are computed
  on structure-respecting splits — 39 of its datasets declare a group or time column.

**Next**: [notebook 07](https://colab.research.google.com/github/Innixma/kdd2026_tutorial_materials/blob/main/notebooks/07_timeseries_forecasting.ipynb) closes the loop by applying the in-context idea to time series forecasting.